In [3]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [ ]:
# ======================================
# FINAL PREPROCESSING (STABLE + STRONG)
# ======================================

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

DATASET_PATH = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
OUTPUT_ROOT = "/kaggle/working/golden_frames_final_v3"

SAMPLE_FPS = 1
THRESHOLD_PERCENTILE = 70
VIDEOS_PER_FOLDER = 1000
MAX_FRAMES = 20            

TOTAL_GOLDEN_FRAMES = 0

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# ======================================
# MEDIAPIPE
# ======================================

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True
)

LEFT_EYE = [33,133,160,159,158,144,145,153]
RIGHT_EYE = [362,263,387,386,385,373,374,380]
MOUTH = [13,14,78,308,82,312,87,317]

IMPORTANT_IDX = LEFT_EYE + RIGHT_EYE + MOUTH

def get_landmarks(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    res = face_mesh.process(rgb)

    if not res.multi_face_landmarks:
        return None

    h, w = frame.shape[:2]
    pts = []

    for idx in IMPORTANT_IDX:
        lm = res.multi_face_landmarks[0].landmark[idx]
        pts.append([lm.x*w, lm.y*h])

    return np.array(pts, dtype=np.float32)

# ======================================
# JITTER ONLY (NO FLOW)
# ======================================

def compute_jitter(prev_pts, curr_pts):
    if prev_pts is None or curr_pts is None:
        return None

    diff = np.linalg.norm(prev_pts - curr_pts, axis=1)
    return float(np.median(diff))

# ======================================
# MAIN
# ======================================

def extract_golden_frames(video_path):
    global TOTAL_GOLDEN_FRAMES

    label = "real" if "original" in video_path else "fake"
    name = os.path.splitext(os.path.basename(video_path))[0]

    save_dir = os.path.join(OUTPUT_ROOT, label, name)
    ensure_dir(save_dir)

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 25

    step = max(int(fps / SAMPLE_FPS), 1)

    prev_pts = None
    scores = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % step != 0:
            frame_idx += 1
            continue

        pts = get_landmarks(frame)
        score = compute_jitter(prev_pts, pts)

        scores.append((frame_idx, score))

        if pts is not None:
            prev_pts = pts

        frame_idx += 1

    cap.release()

    df = pd.DataFrame(scores, columns=["frame", "score"])
    valid = df.dropna()

    if len(valid) == 0:
        return

    thresh = np.percentile(valid["score"], THRESHOLD_PERCENTILE)

    selected = df[df["score"] >= thresh]
    selected = selected.sort_values("score", ascending=False)

    keep = selected["frame"].tolist()[:MAX_FRAMES]

    TOTAL_GOLDEN_FRAMES += len(keep)

    cap = cv2.VideoCapture(video_path)

    for f in keep:
        cap.set(cv2.CAP_PROP_POS_FRAMES, f)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite(os.path.join(save_dir, f"{f:06d}.jpg"), frame)

    cap.release()

# ======================================
# RUN
# ======================================

ensure_dir(OUTPUT_ROOT)

folders = ["original","Deepfakes","DeepFakeDetection"]

for folder in folders:
    path = os.path.join(DATASET_PATH, folder)
    if not os.path.exists(path):
        continue

    vids = [v for v in os.listdir(path) if v.endswith(".mp4")]

    print(f"\nProcessing {folder}")

    for v in tqdm(vids[:VIDEOS_PER_FOLDER]):
        extract_golden_frames(os.path.join(path, v))

print("\nDONE")
print(f"\n🔥 TOTAL FRAMES: {TOTAL_GOLDEN_FRAMES}")

2026-03-03 08:57:01.612078: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772528222.006652      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772528222.115794      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772528223.107631      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772528223.107700      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772528223.107703      55 computation_placer.cc:177] computation placer alr


Processing original


  0%|          | 0/150 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 150/150 [12:44<00:00,  5.10s/it]



Processing Deepfakes


100%|██████████| 150/150 [12:26<00:00,  4.98s/it]



Processing DeepFakeDetection


100%|██████████| 150/150 [16:05<00:00,  6.44s/it]


DONE

🔥 TOTAL FRAMES: 3264


In [15]:
# import shutil
# shutil.make_archive('/kaggle/working/golden_frames_final_v3', 'zip', '/kaggle/working')
import os

file_path = "/kaggle/working/golden_frames_final_v3.zip"

if os.path.exists(file_path):
    os.remove(file_path)
    print("Zip file deleted successfully ✅")
else:
    print("File does not exist ❌")

Zip file deleted successfully ✅
